# Week 8 — Encoder Ablation Analysis

Builds the 5×4 agent×encoder performance matrix and answers the four research questions:

1. Does recurrent integration consistently outperform snapshot encoders?
2. Is the recurrent advantage largest in high-vol / trending regimes?
3. Does the recurrent advantage compound with the CVaR distributional objective?
4. Does AE pre-training accelerate convergence vs CNN trained from scratch?

Also runs latent space PCA on the AE encoder.

**Inputs:** `logs/` (all 17 variant run directories)  
**Outputs:** `experiments/w07_ablation/`

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import torch

from evaluation.ablation import AblationAnalysis
from evaluation.visualize import Visualizer
from training.evaluate import load_agent, evaluate_checkpoint
from envs.lob_env import LOBMarketMakingEnv

LOG_ROOT  = Path('../logs')
CKPT_ROOT = Path('../checkpoints')
EXP_DIR   = Path('../experiments/w07_ablation')
EXP_DIR.mkdir(parents=True, exist_ok=True)

REGIME = 'high_vol'
REWARD = 'asymmetric'
SEED   = 42

ab  = AblationAnalysis(log_root=LOG_ROOT, regime=REGIME, reward=REWARD, seed=SEED)
viz = Visualizer(log_root=LOG_ROOT, ckpt_root=CKPT_ROOT, out_root=EXP_DIR)
print('Setup complete')

## 0 — Missing runs report

In [ ]:
missing = ab.missing_runs()
if missing:
    print(f'{len(missing)} runs not yet completed:')
    for tag in missing:
        print(f'  {tag}')
    print()
    print('Submit these to HPC before running the full ablation.')
else:
    print('All 17 variants complete — ready for full ablation.')

## 1 — Ablation matrix heatmaps

In [ ]:
# Primary metric: Sharpe
sharpe_matrix = ab.build_matrix(metric='sharpe')
print('Sharpe matrix:')
display(sharpe_matrix.style.format('{:.4f}').background_gradient(cmap='RdYlGn', axis=None))
sharpe_matrix.to_csv(EXP_DIR / 'ablation_sharpe.csv')

In [ ]:
# All metrics
all_matrices = ab.build_matrix_all_metrics()
for metric, mat in all_matrices.items():
    mat.to_csv(EXP_DIR / f'ablation_{metric}.csv')
    print(f'{metric}:')
    display(mat.style.format('{:.4f}').background_gradient(
        cmap='RdYlGn' if metric not in ('map', 'mdd') else 'RdYlGn_r',
        axis=None
    ))
    print()

In [ ]:
# Heatmap figure
viz.plot_ablation_matrix(
    metric=  'sharpe',
    regime=  REGIME,
    seed=    SEED,
    save=    True,
)

## 2 — Q1: Recurrent vs snapshot advantage

In [ ]:
print('Q1: Does recurrent consistently outperform snapshot encoders?')
print('─' * 60)

rec_df = ab.recurrent_advantage(metric='sharpe')
display(rec_df)
rec_df.to_csv(EXP_DIR / 'recurrent_advantage.csv', index=False)

n_wins = rec_df['recurrent_wins'].sum()
n_total = len(rec_df)
print(f'\nRecurrent wins: {n_wins}/{n_total} agents')

if n_wins == n_total:
    print('RESULT: Recurrent CONSISTENTLY outperforms snapshot across all agents.')
elif n_wins > n_total / 2:
    print('RESULT: Recurrent outperforms for MOST agents but not all.')
    winners   = rec_df[rec_df['recurrent_wins']]['agent'].tolist()
    non_wins  = rec_df[~rec_df['recurrent_wins']]['agent'].tolist()
    print(f'  Recurrent wins:  {winners}')
    print(f'  Snapshot better: {non_wins}')
else:
    print('RESULT: Snapshot encoders are competitive with recurrent.')
    print('  Recurrent integration may not be justified by these results.')

## 3 — Q2: Recurrent advantage by regime

In [ ]:
print('Q2: Is recurrent advantage largest in high-vol / trending regimes?')
print('─' * 65)
print('Hypothesis: temporal order flow clustering is more predictive')
print('in high-vol and trending → recurrent advantage should be larger.')
print()

regime_df = ab.recurrent_advantage_by_regime(
    regimes=['low_vol', 'high_vol', 'trending', 'normal'],
    agent='qrdqn',
    metric='sharpe',
)
display(regime_df)
regime_df.to_csv(EXP_DIR / 'recurrent_by_regime.csv', index=False)

if not regime_df.empty and 'advantage' in regime_df.columns:
    best_regime = regime_df.iloc[0]['regime']
    print(f'\nLargest recurrent advantage in: {best_regime}')
    if best_regime in ('high_vol', 'trending'):
        print('RESULT: Confirms hypothesis — temporal features most useful in volatile regimes.')
    else:
        print('RESULT: Hypothesis NOT confirmed — recurrent advantage is regime-agnostic.')

## 4 — Q3: Distributional advantage (QR-DQN/IQN vs DQN/PPO)

In [ ]:
print('Q3: Does CVaR distributional objective add value over vanilla DQN?')
print('─' * 65)

for enc in ['handcrafted', 'cnn', 'autoencoder']:
    print(f'\nEncoder: {enc}')
    dist_df = ab.distributional_advantage(metric='sharpe', encoder=enc)
    display(dist_df[['agent', 'distributional', 'sharpe']])

print()
print('Recurrent variant:')
dist_rec = ab.distributional_advantage(metric='sharpe', encoder='handcrafted', recurrent=True)
display(dist_rec[['agent', 'distributional', 'sharpe']])

## 5 — Q4: AE pre-training vs CNN convergence speed

In [ ]:
print('Q4: Does AE pre-training accelerate convergence vs CNN from scratch?')
print('─' * 65)

conv_df = ab.ae_vs_cnn_convergence(agent='qrdqn', metric='sharpe')

if not conv_df.empty:
    from evaluation.visualize import THEME, AGENT_COLORS, _dark_fig, _label, _legend
    import numpy as np

    fig, ax = _dark_fig(figsize=(10, 4))
    colors  = {'autoencoder': '#06b6d4', 'cnn': '#f59e0b'}

    for enc, grp in conv_df.groupby('encoder'):
        grp = grp.sort_values('episode')
        vals = grp['sharpe'].values
        smooth = np.convolve(vals, np.ones(20)/20, mode='valid')
        ax.plot(np.arange(len(smooth)) + 20, smooth,
                color=colors.get(enc, '#ffffff'), lw=2, label=enc)

    _label(ax, xlabel='Episode', ylabel='Sharpe (smoothed 20-ep)',
           title='Q4: AE pre-training vs CNN from scratch — convergence speed')
    _legend(ax)
    path = EXP_DIR / 'ae_vs_cnn_convergence.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=THEME['bg'])
    import matplotlib.pyplot as plt; plt.close(fig)
    print(f'Saved → {path}')
else:
    print('No AE/CNN data — run encoder ablation first')

## 6 — AE latent space PCA

In [ ]:
AE_CKPT = Path('../checkpoints/ae_encoder_16.pt')

if not AE_CKPT.exists():
    print(f'AE checkpoint not found: {AE_CKPT}')
    print('Run: python training/pretrain_ae.py --latent_dim 16')
else:
    from encoders.autoencoder import AEEncoder
    import torch

    ae = AEEncoder.from_checkpoint(AE_CKPT)
    ae.eval()

    # Load some LOB snapshots and run through encoder
    snap_path = Path('../data/processed/lob_snapshots.npy')
    if snap_path.exists():
        snaps = np.load(str(snap_path)).astype(np.float32)

        # Take a subset for PCA
        idx    = np.random.choice(len(snaps), min(2000, len(snaps)), replace=False)
        subset = torch.from_numpy(snaps[idx])

        with torch.no_grad():
            z = ae(subset).numpy()   # (N, 16)

        # Synthetic labels (in real use: label by regime/time-of-day)
        labels     = np.zeros(len(z), dtype=int)
        label_names = ['all data']

        viz.plot_latent_space_pca(
            latent_vectors = z,
            labels         = labels,
            label_names    = label_names,
            save           = True,
        )
        print(f'Latent space shape: {z.shape}')
        print(f'Latent norm (mean): {np.linalg.norm(z, axis=1).mean():.4f}')
    else:
        print(f'Snapshots not found: {snap_path}')
        print('Run data/process_lobster.py first')

## 7 — Full ablation summary

In [ ]:
print(ab.summary(metric='sharpe'))

## 8 — Write-up claims check

In [ ]:
# Auto-generate write-up claim statements from the results
# Fill in after results are available

matrix = ab.build_matrix(metric='sharpe')

if not matrix.empty and not matrix.isna().all().all():
    best_val   = float(matrix.values[~np.isnan(matrix.values)].max())
    best_loc   = matrix.stack().idxmax()
    best_agent = best_loc[0]
    best_enc   = best_loc[1]

    rec_df = ab.recurrent_advantage(metric='sharpe')
    rec_wins = rec_df['recurrent_wins'].sum() if not rec_df.empty else 0

    print('CLAIM STATEMENTS (fill numbers once results are available):')
    print()
    print(f'C1: The best-performing variant is {best_agent} + {best_enc}')
    print(f'    with final Sharpe = {best_val:.4f}.')
    print()
    print(f'C2: Recurrent integration outperforms the best snapshot encoder')
    print(f'    for {rec_wins}/{len(rec_df)} agents, with the largest advantage')
    print(f'    in the high-vol regime (see Table X).')
    print()
    print(f'C3: QR-DQN and IQN (distributional) outperform DQN (non-distributional)')
    print(f'    on Sharpe across all encoder variants, confirming that CVaR-objective')
    print(f'    training improves both risk-adjusted return and inventory management.')
else:
    print('Run ablation training first to generate claims.')